## Creates matched_phrases table based on annotated verbs and deprel 

### table for later filtering based on semantic role


In [1]:
import sqlite3
import pandas as pd
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [2]:
DB_DIR = "../example_data"

# verbimustrite andmebaas
PATTERN_DB = f"{DB_DIR}/verb_patterns.db"

# transaktsioonide andmebaas
TRANSACTIONS_DB = f"{DB_DIR}/transactions.db"

# enriched transaktsioonide andmebaas
ENRICHED_TRANSACTIONS_DB = f"{DB_DIR}/enriched_transactions.db"

# Siia salvestuvad loodavad tabelid
PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"

ENRICHED_TRANSACTIONS_TABLE = "transaction_row"
PATTERNS_TABLE = "patterns"
VERB_MATCHES = "verb_matches"
SEMANTIC_ANN = "semantic_annotations"
# result table
MATCHES_TABEL = "matched_phrases"

# new patterns table with semantic role, temporary
PATTERNS_WITH_ROLE = "patterns_role"
# temporary table for pattern-join-transaction_head
PATTERNS_HEAD = "patterns_tr_head"

# role condition
# can be included for the final table
#ROLE_CONDITION = 'isik_alati'

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTIONS_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTIONS_DB}" AS entrans')
cur.execute(f'ATTACH DATABASE "{PATTERN_DB}" AS pat')

## Workflow



### Add semantic role to patterns

In [4]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERNS_WITH_ROLE))

cur.execute("""
CREATE TABLE {tbl1} AS
SELECT DISTINCT
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.verb_compound as verb_compound,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.deprel as deprel,
    semantic_role ||'_'||certainty as role
FROM 
    {pattbl} as pat
INNER JOIN 
    {semann} as ann
ON
    pat.pat_id = ann.pattern_id

""".format(tbl1=PATTERNS_WITH_ROLE, pattbl=PATTERNS_TABLE, semann=SEMANTIC_ANN))

CPU times: user 50 ms, sys: 14.9 ms, total: 64.9 ms
Wall time: 75 ms


In [5]:
display_db_table(con, PATTERNS_WITH_ROLE)

,pat_id,verb_word,verb_compound,phrase_nr,phrase_case,deprel,role
0,1,saama,,1,abl,obl,isik_vahel
1,1,saama,,1,abl,obl,koht_vahel
2,1,saama,,1,abl,obl,muu_mitte kunagi
3,2,tulema,,1,abl,obl,isik_vahel
4,2,tulema,,1,abl,obl,koht_vahel


### temp table for join

In [6]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERNS_HEAD))

cur.execute("""
CREATE TABLE {tbl1} AS
SELECT DISTINCT
    head.id as head_id,
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.verb_compound as verb_compound,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.deprel as pat_deprel,
    pat.role as sem_role
FROM 
    {pattbl} as pat
INNER JOIN 
    trans.transaction_head as head
ON
    pat.verb_word = head.verb

""".format(tbl1=PATTERNS_HEAD, pattbl=PATTERNS_WITH_ROLE))

CPU times: user 457 ms, sys: 139 ms, total: 595 ms
Wall time: 609 ms


In [7]:
display_db_table(con, PATTERNS_HEAD)

,head_id,pat_id,verb_word,verb_compound,phrase_nr,phrase_case,pat_deprel,sem_role
0,3,1,saama,,1,abl,obl,isik_vahel
1,44,1,saama,,1,abl,obl,isik_vahel
2,54,1,saama,,1,abl,obl,isik_vahel
3,74,1,saama,,1,abl,obl,isik_vahel
4,96,1,saama,,1,abl,obl,isik_vahel


### matched_phrases

Praegu on joini aluseks head_id, deprel, feats (vajadusel ka semantic_role). 

` NB!!! patterns tabeli phrase_case on abl/all/ad jne ja neid tulebks matchida feats veerus olevaga`

In [8]:
%%time 

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=MATCHES_TABEL))


cur.execute("""
CREATE TABLE {new_table} AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    --tr.id as transaction_id,
    --tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tbl1.verb_compound as verb_compound,
    tr.lemma as root_word,
    tbl1.pat_deprel as deprel,
    tbl1.phrase_case as phrase_case,
    tbl1.sem_role as semantic_role,
    tr.koht as koht,
    elus as elus
    
FROM 
    {pattbl} as tbl1
JOIN 
    entrans.{trans_tbl} as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0
""".format(new_table=MATCHES_TABEL, pattbl=PATTERNS_HEAD, trans_tbl=ENRICHED_TRANSACTIONS_TABLE))

CPU times: user 130 ms, sys: 7.11 ms, total: 137 ms
Wall time: 146 ms


In [9]:
display_db_table(con, MATCHES_TABEL)

,head_id,pat_id,verb_word,verb_compound,root_word,deprel,phrase_case,semantic_role,koht,elus
0,179,4,nõudma,,mina,obl,abl,isik_alati,,YES
1,179,4,nõudma,,mina,obl,abl,koht_mitte kunagi,,YES
2,179,4,nõudma,,mina,obl,abl,muu_mitte kunagi,,YES
3,656,116,kaotama,,vana,obl,abl,isik_mitte kunagi,,
4,656,116,kaotama,,vana,obl,abl,koht_mitte kunagi,,


In [10]:
con.close()